# Hafta 2 — Kaggle Uretim (S / M1 / M2)

Colab duman testi bu notebook'un girdisidir; burada **tekrar olcum yapilmaz**,
dogrudan uretim yapilir. Olculen hizlar (Colab T4):

| bilesen | sn/goruntu | tepe VRAM |
|---|---|---|
| sd15 | 30.0 | 1.9 GB |
| sdxl | 222.6 | 5.5 GB |
| sd_turbo | 26.7 | 1.8 GB |
| inpaint (sdxl) | 111.5 | 5.8 GB |

**NEDEN KAGGLE:** ucretsiz Colab'in sistem RAM'i ~13 GB ve flux_schnell
(24 GB agirlik) orada oturumu cokertti. Kaggle 32 GB RAM + 30 sa/hafta kota
+ 9-12 sa oturum veriyor, ustelik **T4 x2** ile iki uretim paralel kosuyor.

## Ayarlar — calistirmadan once kontrol et
1. Sag menu **Session options -> Accelerator -> GPU T4 x2**
2. **Internet: On** (model indirme ve git clone icin sart)
3. Veri seti bagli mi: **Add Input -> cardd-data**

Hedef (Dengeli senaryo, ~8.1 saat duvar saati):
`sd15=150, sdxl=80, sd_turbo=100, M1=200, M2=60 SD + 60 klasik`

## 0. Ortam ve GPU

In [ ]:
import subprocess, sys, os, platform, torch
from pathlib import Path

print("Python :", platform.python_version())
print("torch  :", torch.__version__)
print("GPU sayisi:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB")

import psutil
print(f"\nSistem RAM: {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"Disk (/kaggle/working): {psutil.disk_usage('/kaggle/working').free/1e9:.0f} GB bos")

if torch.cuda.device_count() < 2:
    print("\n>>> UYARI: 2 GPU yok. Session options -> Accelerator -> GPU T4 x2 sec.")
    print(">>> Tek GPU ile de calisir ama paralel plan ise yaramaz, sure ~2 katina cikar.")

In [ ]:
!pip install -q diffusers accelerate safetensors transformers sentencepiece protobuf
import diffusers; print("diffusers:", diffusers.__version__)

## 1. Repo ve veri

Veri `/kaggle/input` altinda **salt-okunur** bagli ve Kaggle zip'i otomatik
acmis durumda -- Colab'daki gibi 3 GB'i diske acmaya gerek yok. Kodun
bekledigi `data/raw/cardd` yoluna sembolik bag kuruyoruz.

In [ ]:
REPO_URL = "https://github.com/Tunahan-46/insurance-image-forensics.git"
WORK = Path("/kaggle/working/insurance-image-forensics")

if not WORK.exists():
    !git clone -q {REPO_URL} {WORK}
else:
    !git -C {WORK} pull -q
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())

# Veri setinin gercek adini bul (Kaggle bazen adi normalize eder)
INPUT = Path("/kaggle/input")
candidates = [d for d in INPUT.iterdir() if d.is_dir()] if INPUT.exists() else []
print("\nBagli veri setleri:", [d.name for d in candidates])

DATA = None
for d in candidates:
    if (d / "CarDD_COCO").exists() or list(d.rglob("CarDD_COCO"))[:1]:
        DATA = d
        break
print("Kullanilacak:", DATA)
assert DATA is not None, "cardd-data bagli degil. Add Input -> cardd-data"


In [ ]:
# CarDD'yi kodun bekledigi yere BAGLA (kopyalama yok)
CARDD = WORK / "data/raw/cardd"
CARDD.mkdir(parents=True, exist_ok=True)

coco_src = next(DATA.rglob("CarDD_COCO"))
if not (CARDD / "CarDD_COCO").exists():
    os.symlink(coco_src, CARDD / "CarDD_COCO")

# SOD maskeleri: zip duz yapida (CarDD-TR-Mask, ...), kod bunlari
# CarDD_SOD/CarDD-XX/CarDD-XX-Mask altinda arar.
SOD = CARDD / "CarDD_SOD"
for split in ["TR", "VAL", "TE"]:
    dst_dir = SOD / f"CarDD-{split}"
    dst = dst_dir / f"CarDD-{split}-Mask"
    if dst.exists():
        continue
    hits = list(DATA.rglob(f"CarDD-{split}-Mask"))
    if not hits:
        print(f"  UYARI: CarDD-{split}-Mask bulunamadi")
        continue
    dst_dir.mkdir(parents=True, exist_ok=True)
    os.symlink(hits[0], dst)

for p in ["CarDD_COCO/train2017", "CarDD_COCO/val2017", "CarDD_COCO/test2017",
          "CarDD_SOD/CarDD-TR/CarDD-TR-Mask", "CarDD_SOD/CarDD-TE/CarDD-TE-Mask"]:
    d = CARDD / p
    n = len(list(d.iterdir())) if d.exists() else "YOK"
    print(f"  {p:<40} {n}")

## 2. Uretim manifesti

Kendi telefon fotograflarin ve M3 ciktilarin burada yok -- gerekmiyor da.
Split'ler DETERMINISTIK: CarDD'nin bolumu klasor yapisindan geliyor, hash'ten
degil. Yani buradaki manifest, CarDD satirlari icin yereldekiyle **ayni**
split'i verir. Bu manifest sadece uretim girdisi; gercek manifest yerelde
kurulacak.

In [ ]:
!python scripts/build_manifest_v2.py

## 3. Uretim — iki GPU paralel

`nohup ... &` ile iki is ayni anda arka planda baslar:

- **GPU 0** -> S katmani (sd15, sdxl, sd_turbo)   ~6.9 saat
- **GPU 1** -> M katmani (inpaint_add, inpaint_remove)  ~8.1 saat

`CUDA_VISIBLE_DEVICES` her surece tek GPU gosterir, boylece iki is
birbirinin VRAM'ine girmez. Ciktilar `logs/` altina yazilir.

Uretecilerde `resume=True` varsayilan: oturum koparsa ayni hucreyi tekrar
calistir, kaldigi yerden devam eder.

In [ ]:
Path("logs").mkdir(exist_ok=True)

# NOT: "set -e" bilincli olarak YOK ve her komut "|| echo" ile korunuyor.
# Isler birbirinden bagimsiz: sdxl OOM verirse sd_turbo yine de uretilmeli.
# Tek bir hata gece boyu surecek uretimin geri kalanini iptal etmemeli.
GPU0 = '''
python -m src.data.generators.fully_synthetic --model sd15     --n 150 --seed 1 || echo "!!! sd15 BASARISIZ"
python -m src.data.generators.fully_synthetic --model sdxl     --n 80  --seed 2 || echo "!!! sdxl BASARISIZ"
python -m src.data.generators.fully_synthetic --model sd_turbo --n 100 --seed 3 || echo "!!! sd_turbo BASARISIZ"
echo "=== GPU0 BITTI ==="
'''

# M2'nin yarisi klasik OpenCV inpaint: GPU gerektirmez, saniyeler surer ve
# planin istedigi "LaMa + SD-inpaint" cesitliligini saglar. Dedektor sadece
# difuzyon izlerini degil klasik inpaint izlerini de gormeli.
GPU1 = '''
python -m src.data.generators.inpaint_add --manifest data/processed/manifest_v2.parquet --n 200 --model sdxl --seed 11 || echo "!!! inpaint_add BASARISIZ"
python -m src.data.generators.inpaint_remove --manifest data/processed/manifest_v2.parquet --n 60 --method sd_inpaint --model sdxl --seed 12 || echo "!!! inpaint_remove/sd BASARISIZ"
python -m src.data.generators.inpaint_remove --manifest data/processed/manifest_v2.parquet --n 60 --method telea --seed 13 || echo "!!! inpaint_remove/telea BASARISIZ"
echo "=== GPU1 BITTI ==="
'''

import subprocess

# IPython'in "!cmd &" bicimi arka plan surecini kernel'e baglayabilir ve
# hucre bitince oldurebilir. Popen bagimsiz surec baslatir.
PROCS = globals().get("PROCS", {})

for gpu, script in [(0, GPU0), (1, GPU1)]:
    onceki = PROCS.get(gpu)
    if onceki is not None and onceki.poll() is None:
        print(f"GPU {gpu}: zaten calisiyor (pid={onceki.pid}), tekrar baslatilmadi.")
        continue
    sh = Path(f"logs/gpu{gpu}.sh")
    sh.write_text(script)
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    log = open(f"logs/gpu{gpu}.log", "a")
    PROCS[gpu] = subprocess.Popen(
        ["bash", str(sh)], env=env, stdout=log, stderr=subprocess.STDOUT, cwd=str(WORK)
    )
    print(f"GPU {gpu} baslatildi (pid={PROCS[gpu].pid})")

print("\nIlerlemeyi asagidaki hucreyle izle.")
print("Bu hucre tekrar calistirilirsa calisan isler KORUNUR (poll kontrolu).")

### 3b. Ilerleme izleme

Bu hucreyi istedigin kadar calistirabilirsin. `nvidia-smi` iki GPU'nun da
calistigini gostermeli.

In [ ]:
import subprocess, time
from pathlib import Path

def durum():
    print("=" * 70)
    for g in (0, 1):
        log = Path(f"logs/gpu{g}.log")
        satirlar = log.read_text(errors="ignore").strip().splitlines() if log.exists() else []
        son = [s for s in satirlar if s.strip()][-3:]
        print(f"\n--- GPU {g} ---")
        for s in son:
            print("  " + s[:110])
    print("\n--- uretilen dosyalar ---")
    for d, ad in [("data/raw/synthetic", "S"),
                  ("data/raw/manipulated/inpaint_add", "M1"),
                  ("data/raw/manipulated/inpaint_remove", "M2")]:
        p = Path(d)
        n = len([x for x in p.rglob("*.png") if "masks" not in x.parts]) if p.exists() else 0
        print(f"  {ad:<4} {n}")
    print()
    !nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader

durum()

## 4. Ciktilari kalici hale getir

**BU ADIMI ATLAMA.** `/kaggle/working` oturum bitince silinir. Iki secenek:

1. **Save Version** (sag ust) -> `/kaggle/working` iceriginin tamami cikti
   olarak kaydedilir, sonra indirilebilir. En kolayi bu.
2. Asagidaki hucre zip'ler; zip'i notebook ciktilarindan indirirsin.

Uretim BITTIKTEN sonra calistir.

In [ ]:
import shutil
from pathlib import Path

for folder, ad in [("data/raw/synthetic", "synthetic"),
                   ("data/raw/manipulated", "manipulated")]:
    p = Path(folder)
    if not p.exists():
        print(f"{folder}: yok"); continue
    n = len(list(p.rglob("*.png")))
    hedef = f"/kaggle/working/w2_{ad}"
    shutil.make_archive(hedef, "zip", p)
    mb = Path(hedef + ".zip").stat().st_size / 1e6
    print(f"{folder}: {n} dosya -> {hedef}.zip ({mb:.0f} MB)")

print("\nSonraki adim (YERELDE):")
print("  1. zipleri indir, data/raw/ altina ac")
print("  2. python scripts/build_manifest_v2.py    # split 2854/817/381 kalmali")
print("  3. python scripts/apply_laundering.py")
print("  4. python scripts/run_e1_shortcut.py      # Gorev A artik olculebilir")